In [5]:
# 초록에서 키워드 추출, 영어는 소문자로 통일

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
target_csv_path = 'keyword_missing_abstract_being_papers.csv'  # 입력 파일
rule_map_path = 'keyword_rule_map.json'                        # 룰맵 파일
output_csv_path = 'extract_keywords_from_abstract.csv'         # 결과 파일

# =============================================================================
# 2. 룰맵 로딩 및 정규화
# =============================================================================
print("1. 키워드 룰맵 로드 및 정규화(공백제거) 중...")

normalized_rule_map = {} 

try:
    with open(rule_map_path, 'r', encoding='utf-8') as f:
        rule_data = json.load(f)

    # 리스트나 딕셔너리 구조에 맞춰 유연하게 로드
    items = []
    if isinstance(rule_data, dict):
        if "exact_map" in rule_data:
            for v, c in rule_data["exact_map"].items():
                norm_key = v.replace(" ", "").lower()
                normalized_rule_map[norm_key] = c
        # 추가 리스트가 있다면
        if "variants_list" in rule_data:
            items = rule_data["variants_list"]
    elif isinstance(rule_data, list):
        items = rule_data

    for item in items:
        if "variant_norm" in item and "chosen" in item:
            norm_key = str(item["variant_norm"]).replace(" ", "").lower()
            normalized_rule_map[norm_key] = item["chosen"]

    print(f"   -> 룰맵 준비 완료: {len(normalized_rule_map)}개 항목")

except FileNotFoundError:
    print(f"❌ 오류: 룰맵 파일({rule_map_path})을 찾을 수 없습니다.")
    exit()

# =============================================================================
# 3. 조사 처리 및 영문 소문자 변환 로직이 추가된 추출 함수
# =============================================================================
# 흔히 붙는 조사 목록 (끝 글자 기준)
COMMON_JOSA = ['은', '는', '이', '가', '을', '를', '의', '에', '와', '과', '로', '으로', '도', '만', '서']

def extract_keywords_with_josa_handling(text, rule_map, max_ngram=5):
    """
    텍스트에서 룰맵 기반으로 키워드를 추출합니다.
    - 조사 제거 로직 포함
    - 추출된 키워드는 영어일 경우 소문자로 변환, 한글은 유지
    """
    if not isinstance(text, str) or not text:
        return ""

    # 특수문자 제거 (한글, 영문, 숫자 외 공백 처리)
    clean_text = re.sub(r'[^\w\s가-힣]', ' ', text)
    words = clean_text.split()
    
    found_keywords = set()
    n = len(words)
    i = 0
    
    while i < n:
        match_found = False
        
        # Longest Match (긴 단어 조합부터 검사)
        for length in range(min(max_ngram, n - i), 0, -1):
            chunk_words = words[i : i+length]
            base_chunk_str = "".join(chunk_words).lower() # 검색키: 공백 제거+소문자
            
            # --- 매칭 확인 내부 함수 ---
            def check_and_add(search_key):
                if search_key in rule_map:
                    # 1. 룰맵에서 표준어(chosen) 가져오기
                    standard_word = rule_map[search_key]
                    
                    # 2. [수정됨] 소문자 변환 로직
                    #    - 영어라면 소문자로 변환됨 ("Artificial Intelligence" -> "artificial intelligence")
                    #    - 한글이라면 그대로 유지됨 ("인공지능" -> "인공지능")
                    final_word = standard_word.lower()
                    
                    found_keywords.add(final_word)
                    return True
                return False
            # -------------------------

            # 1. 정확히 매칭되는지 확인 (예: "deeplearning", "인공지능")
            if check_and_add(base_chunk_str):
                i += length
                match_found = True
                break
            
            # 2. 한글 조사 제거 후 확인 (예: "인공지능을" -> "인공지능")
            if len(base_chunk_str) > 1: # 최소 2글자 이상일 때만
                last_char = base_chunk_str[-1]
                
                # 일반적인 1글자 조사 처리
                if last_char in COMMON_JOSA:
                    stripped_str = base_chunk_str[:-1]
                    if check_and_add(stripped_str):
                        i += length
                        match_found = True
                        break
                
                # '으로' 같은 2글자 조사 처리
                if len(base_chunk_str) > 2 and base_chunk_str.endswith('으로'):
                     stripped_str = base_chunk_str[:-2]
                     if check_and_add(stripped_str):
                        i += length
                        match_found = True
                        break

        if not match_found:
            i += 1
            
    return ",".join(list(found_keywords))

# =============================================================================
# 4. 실행 및 저장
# =============================================================================
print("2. 논문 초록에서 키워드 추출 (조사 처리 + 영문 소문자화)...")

if os.path.exists(target_csv_path):
    df = pd.read_csv(target_csv_path)
    
    # NaN 처리
    df['ABST_KR'] = df['ABST_KR'].fillna("")
    df['ABST_EN'] = df['ABST_EN'].fillna("")
    
    extracted_results = []
    
    # 진행 상황 표시용
    total_rows = len(df)
    
    for idx, row in df.iterrows():
        # 국문과 영문을 모두 사용하여 매칭 확률 높임
        full_text = str(row['ABST_KR']) + " " + str(row['ABST_EN'])
        
        # 키워드 추출
        kwd = extract_keywords_with_josa_handling(full_text, normalized_rule_map)
        extracted_results.append(kwd)
        
        if (idx + 1) % 1000 == 0:
            print(f"   -> {idx + 1}/{total_rows} 처리 중...")
        
    df['EXTRACTED_KEYWORDS'] = extracted_results
    
    # CSV 저장
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    
    # 결과 통계
    non_empty = df[df['EXTRACTED_KEYWORDS'] != ""]
    print("-" * 50)
    print(f"✅ 추출 및 변환 완료!")
    print(f"   - 전체 대상: {len(df)}건")
    print(f"   - 키워드 발견 성공: {len(non_empty)}건")
    print(f"   - 저장 파일: {output_csv_path}")
    print("-" * 50)
    print("미리보기 (상위 3건):")
    # 결과 확인 시, 영문이 소문자로 잘 바뀌었는지 확인 가능
    print(df[['NODE_ID', 'EXTRACTED_KEYWORDS']].head(3))

else:
    print(f"❌ 오류: 입력 파일({target_csv_path})이 없습니다.")

1. 키워드 룰맵 로드 및 정규화(공백제거) 중...
   -> 룰맵 준비 완료: 1931개 항목
2. 논문 초록에서 키워드 추출 (조사 처리 + 영문 소문자화)...
--------------------------------------------------
✅ 추출 및 변환 완료!
   - 전체 대상: 679건
   - 키워드 발견 성공: 575건
   - 저장 파일: extract_keywords_from_abstract.csv
--------------------------------------------------
미리보기 (상위 3건):
        NODE_ID                                 EXTRACTED_KEYWORDS
0  NODE10610913               railway technology,track,temperature
1  NODE10584495  concrete,resilience,flow,behavior,durability,m...
2  NODE10584498                     reinforced concrete,plasticity


In [2]:
# 추출된 한/영 키워드 확인 및 저장

import pandas as pd
import os
from collections import Counter

# 1. 파일 경로 설정
input_csv_path = 'extract_keywords_from_abstract.csv'
output_csv_path = 'keyword_statistics.csv'  # 저장할 파일 이름

# 2. 데이터 로드 및 키워드 집계
print("📂 키워드 추출 결과를 분석합니다...")

if os.path.exists(input_csv_path): # 파일 존재 여부 확인 (os 모듈 사용 권장)
    df = pd.read_csv(input_csv_path)
    
    # NaN 값을 빈 문자열로 처리
    df['EXTRACTED_KEYWORDS'] = df['EXTRACTED_KEYWORDS'].fillna("")
    
    # 모든 키워드를 하나의 리스트로 모으기
    all_keywords = []
    
    for kwd_str in df['EXTRACTED_KEYWORDS']:
        if str(kwd_str).strip(): # 빈 칸이 아닌 경우만 (문자열 변환 안전장치 추가)
            # 쉼표(,)로 구분된 키워드들을 분리해서 리스트에 추가
            keywords = [k.strip() for k in str(kwd_str).split(",")]
            all_keywords.extend(keywords)
            
    # 개수 세기 (Counter 활용)
    keyword_counts = Counter(all_keywords)
    
    # 3. 콘솔 출력 (상위 20개 확인용)
    print(f"\n✅ 총 추출된 키워드 수 (중복 포함): {len(all_keywords)}개")
    print(f"✅ 발견된 고유 키워드 종류: {len(keyword_counts)}개")
    print("-" * 45)
    print(f"{'순위':<5} {'키워드':<30} {'개수':<5}")
    print("-" * 45)
    
    # 상위 20개 출력
    for rank, (keyword, count) in enumerate(keyword_counts.most_common(20), 1):
        print(f"{rank:<5} {keyword:<30} {count:<5}")
        
    print("-" * 45)
    
    # -----------------------------------------------------------
    # 4. [추가됨] 전체 결과 파일 저장 코드
    # -----------------------------------------------------------
    # Counter 객체를 DataFrame으로 변환 (빈도수 내림차순 정렬됨)
    result_df = pd.DataFrame(keyword_counts.most_common(), columns=['Keyword', 'Count'])
    
    # CSV로 저장 (utf-8-sig를 써야 엑셀에서 한글이 안 깨짐)
    result_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    
    print(f"\n💾 전체 키워드 통계가 '{output_csv_path}' 파일로 저장되었습니다.")
    print("   (엑셀에서 열어서 확인해보세요!)")

else:
    print(f"❌ 오류: '{input_csv_path}' 파일이 없습니다. 이전 단계를 먼저 실행해주세요.")

📂 키워드 추출 결과를 분석합니다...

✅ 총 추출된 키워드 수 (중복 포함): 1729개
✅ 발견된 고유 키워드 종류: 310개
---------------------------------------------
순위    키워드                            개수   
---------------------------------------------
1     performance                    96   
2     machine learning               91   
3     artificial intelligence        84   
4     research trends                60   
5     efficiency                     57   
6     internet of things             45   
7     deep learning                  39   
8     content                        35   
9     blockchain                     32   
10    metaverse                      32   
11    6g                             32   
12    autonomous driving             30   
13    optimization                   27   
14    ict                            25   
15    covid 19                       24   
16    generation                     24   
17    5g                             22   
18    network                        21   
19    recognition

In [3]:
# 추출된 한글 키워드 확인

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
result_csv_path = 'extract_keywords_from_abstract.csv' # 결과 파일
rule_map_path = 'keyword_rule_map.json' # 룰맵 파일

# =============================================================================
# 2. 데이터 로드 및 룰맵 준비
# =============================================================================
print("1. 데이터 및 룰맵 로드 중...")

# [룰맵 로드]
normalized_korean_rules = {} # 한글 규칙만 저장 { '인공지능': 'machine learning' }

with open(rule_map_path, 'r', encoding='utf-8') as f:
    rule_data = json.load(f)

# 룰맵 파싱 (이전과 동일 로직)
items = []
if isinstance(rule_data, dict):
    if "exact_map" in rule_data:
        for v, c in rule_data["exact_map"].items():
            # 한글이 포함된 키만 필터링
            if re.search('[가-힣]', v):
                norm_key = v.replace(" ", "").lower()
                normalized_korean_rules[norm_key] = c
    if "variants_list" in rule_data:
        items = rule_data["variants_list"]
elif isinstance(rule_data, list):
    items = rule_data

for item in items:
    if "variant_norm" in item and "chosen" in item:
        v = str(item["variant_norm"])
        if re.search('[가-힣]', v):
            norm_key = v.replace(" ", "").lower()
            normalized_korean_rules[norm_key] = item["chosen"]

print(f"   -> 검증할 한글 규칙 수: {len(normalized_korean_rules)}개")

# [결과 파일 로드]
if os.path.exists(result_csv_path):
    df = pd.read_csv(result_csv_path)
    df['ABST_KR'] = df['ABST_KR'].fillna("")
    df['EXTRACTED_KEYWORDS'] = df['EXTRACTED_KEYWORDS'].fillna("")
    
    # =============================================================================
    # 3. 검증 로직 수행
    # =============================================================================
    print("2. 한글 키워드 변환 여부 검증 시작...")
    
    match_count = 0
    sample_logs = []
    
    # 흔한 조사 제거용 정규식 (간단 버전)
    josa_pattern = re.compile(r'(은|는|이|가|을|를|의|에|로|으로|와|과|도|만|서)$')

    for idx, row in df.iterrows():
        abst_kr = str(row['ABST_KR'])
        extracted = str(row['EXTRACTED_KEYWORDS'])
        
        if not abst_kr.strip(): continue # 국문 초록 없으면 패스
        
        # 정규화된 초록 (공백 제거) - 검색 편의상
        norm_abst = re.sub(r'\s+', '', abst_kr)
        
        # 한글 룰 하나씩 대조 (속도 위해 일부만 샘플링하거나 전체 루프)
        # 여기서는 정확성을 위해 루프를 돌지만, 데이터가 많으면 오래 걸릴 수 있음
        
        row_hit = False
        for kr_key, en_val in normalized_korean_rules.items():
            # 초록에 한글 키워드가 포함되어 있는지 확인 (단순 포함 관계)
            # 주의: "인공지능"이 "인공지능을" 안에 포함되므로 in 연산자로 확인 가능
            if kr_key in norm_abst:
                # 결과에 영문 변환값이 있는지 확인
                if en_val in extracted:
                    match_count += 1
                    row_hit = True
                    
                    # 로그용 샘플 저장 (최대 10개)
                    if len(sample_logs) < 10:
                        sample_logs.append({
                            'NODE_ID': row['NODE_ID'],
                            'Found_Korean': kr_key,
                            'Converted_To': en_val,
                            'In_Abstract': (abst_kr[:30] + "..."),
                            'Extracted_Result': extracted
                        })
                    break # 한 행에서 하나라도 찾으면 카운트하고 다음 행으로 (중복 카운트 방지)
    
    # =============================================================================
    # 4. 결과 출력
    # =============================================================================
    print("-" * 60)
    print(f"✅ 검증 완료!")
    print(f"   - 한글 단어가 포함된 논문 중 변환 성공한 케이스: {match_count}건")
    print("-" * 60)
    print("[변환 성공 샘플 미리보기]")
    print(f"{'Korean(In Abstract)':<20} | {'Converted(English)':<30} | {'Extracted Result'}")
    print("-" * 60)
    
    for log in sample_logs:
        print(f"{log['Found_Korean']:<20} -> {log['Converted_To']:<30} | {log['Extracted_Result'][:30]}...")
        
else:
    print(f"❌ 오류: '{result_csv_path}' 파일이 없습니다.")

1. 데이터 및 룰맵 로드 중...
   -> 검증할 한글 규칙 수: 378개
2. 한글 키워드 변환 여부 검증 시작...
------------------------------------------------------------
✅ 검증 완료!
   - 한글 단어가 포함된 논문 중 변환 성공한 케이스: 448건
------------------------------------------------------------
[변환 성공 샘플 미리보기]
Korean(In Abstract)  | Converted(English)             | Extracted Result
------------------------------------------------------------
온도                   -> temperature                    | temperature,track,railway tech...
온실가스                 -> decomposition                  | education,decomposition,bim,en...
인공지능                 -> machine learning               | energy saving,machine learning...
공동주택                 -> apartment complex              | recognition,covid 19,indoor ai...
수소                   -> hydrogen                       | hydrogen,artificial intelligen...
인공지능                 -> machine learning               | machine learning,internet of t...
시뮬레이션                -> simulation                     | performan

In [4]:
# 추출되지 않은 논문 5개 확인

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정 (사용자 요청 반영)
# =============================================================================
# 이전 단계에서 생성한 '키워드 없고 초록만 있는' 파일
target_csv_path = 'keyword_missing_abstract_being_papers.csv'
# 룰맵 파일
rule_map_path = 'keyword_rule_map.json'
# 결과 저장 파일
output_csv_path = 'extract_keywords_from_abstract.csv'

# =============================================================================
# 2. 룰맵 로딩 및 정규화
# =============================================================================
print("1. 키워드 룰맵 로드 및 정규화(공백제거) 중...")

normalized_rule_map = {} 

with open(rule_map_path, 'r', encoding='utf-8') as f:
    rule_data = json.load(f)

# 리스트나 딕셔너리 구조에 맞춰 유연하게 로드
items = []
if isinstance(rule_data, dict):
    if "exact_map" in rule_data:
        for v, c in rule_data["exact_map"].items():
            norm_key = v.replace(" ", "").lower()
            normalized_rule_map[norm_key] = c
    # 추가 리스트가 있다면
    if "variants_list" in rule_data:
        items = rule_data["variants_list"]
elif isinstance(rule_data, list):
    items = rule_data

for item in items:
    if "variant_norm" in item and "chosen" in item:
        norm_key = str(item["variant_norm"]).replace(" ", "").lower()
        normalized_rule_map[norm_key] = item["chosen"]

print(f"   -> 룰맵 준비 완료: {len(normalized_rule_map)}개 항목")

# =============================================================================
# 3. 조사 처리 로직이 추가된 추출 함수
# =============================================================================
COMMON_JOSA = ['은', '는', '이', '가', '을', '를', '의', '에', '와', '과', '로', '으로', '도', '만', '서']

def extract_keywords_with_josa_handling(text, rule_map, max_ngram=5):
    if not isinstance(text, str) or not text:
        return ""

    # 특수문자 제거 (한글, 영문, 숫자 외 공백 처리)
    clean_text = re.sub(r'[^\w\s가-힣]', ' ', text)
    words = clean_text.split()
    
    found_keywords = set()
    n = len(words)
    i = 0
    
    while i < n:
        match_found = False
        
        # Longest Match (긴 단어 조합부터 검사)
        for length in range(min(max_ngram, n - i), 0, -1):
            chunk_words = words[i : i+length]
            base_chunk_str = "".join(chunk_words).lower() # 공백 제거+소문자
            
            # 1. 정확히 매칭되는지 확인
            if base_chunk_str in rule_map:
                found_keywords.add(rule_map[base_chunk_str])
                i += length
                match_found = True
                break
            
            # 2. 조사 처리 로직
            if len(base_chunk_str) > 1: 
                last_char = base_chunk_str[-1]
                if last_char in COMMON_JOSA:
                    stripped_str = base_chunk_str[:-1] 
                    if stripped_str in rule_map:
                        found_keywords.add(rule_map[stripped_str])
                        i += length
                        match_found = True
                        break
                
                if len(base_chunk_str) > 2 and base_chunk_str.endswith('으로'):
                     stripped_str = base_chunk_str[:-2]
                     if stripped_str in rule_map:
                        found_keywords.add(rule_map[stripped_str])
                        i += length
                        match_found = True
                        break

        if not match_found:
            i += 1
            
    return ",".join(list(found_keywords))

# =============================================================================
# 4. 실행 및 저장, 그리고 실패 샘플 확인
# =============================================================================
print("2. 논문 초록에서 키워드 추출 (조사 처리 적용)...")

if os.path.exists(target_csv_path):
    df = pd.read_csv(target_csv_path)
    
    df['ABST_KR'] = df['ABST_KR'].fillna("")
    df['ABST_EN'] = df['ABST_EN'].fillna("")
    
    extracted_results = []
    
    for idx, row in df.iterrows():
        full_text = str(row['ABST_KR']) + " " + str(row['ABST_EN'])
        kwd = extract_keywords_with_josa_handling(full_text, normalized_rule_map)
        extracted_results.append(kwd)
        
    df['EXTRACTED_KEYWORDS'] = extracted_results
    
    # 저장
    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    
    # 통계 계산
    non_empty = df[df['EXTRACTED_KEYWORDS'] != ""]
    empty_df = df[df['EXTRACTED_KEYWORDS'] == ""] # 추출되지 않은 데이터만 필터링
    
    print("-" * 50)
    print(f"✅ 추출 완료!")
    print(f"   - 전체 대상: {len(df)}건")
    print(f"   - 키워드 발견 성공: {len(non_empty)}건")
    print(f"   - 키워드 미발견: {len(empty_df)}건")
    print(f"   - 저장 파일: {output_csv_path}")
    print("-" * 50)
    
    # 요청하신 실패 샘플 출력 부분
    if len(empty_df) > 0:
        print("\n" + "="*60)
        print(f"❌ [추출 실패 샘플 (상위 5건)]")
        print("   -> 룰맵에 있는 단어가 초록에 없거나 매칭되지 않은 경우입니다.")
        print("="*60)
        
        for i, row in empty_df.head(5).iterrows():
            node_id = row['NODE_ID']
            k_abst = str(row['ABST_KR'])
            e_abst = str(row['ABST_EN'])
            
            # 초록이 너무 길면 잘라서 보여줌
            k_preview = k_abst[:60] + "..." if len(k_abst) > 60 else k_abst
            e_preview = e_abst[:60] + "..." if len(e_abst) > 60 else e_abst
            
            print(f"🔹 NODE_ID: {node_id}")
            print(f"   - 국문: {k_preview}")
            print(f"   - 영문: {e_preview}")
            print("-" * 40)
    else:
        print("🎉 모든 논문에서 키워드가 성공적으로 추출되었습니다!")

else:
    print(f"❌ 오류: 입력 파일({target_csv_path})이 없습니다.")

1. 키워드 룰맵 로드 및 정규화(공백제거) 중...
   -> 룰맵 준비 완료: 1931개 항목
2. 논문 초록에서 키워드 추출 (조사 처리 적용)...
--------------------------------------------------
✅ 추출 완료!
   - 전체 대상: 679건
   - 키워드 발견 성공: 575건
   - 키워드 미발견: 104건
   - 저장 파일: extract_keywords_from_abstract.csv
--------------------------------------------------

❌ [추출 실패 샘플 (상위 5건)]
   -> 룰맵에 있는 단어가 초록에 없거나 매칭되지 않은 경우입니다.
🔹 NODE_ID: NODE10593932
   - 국문: 본 연구에서는 캐나다 오일샌드의 공간적인 분포 경향을 파악하고자 노력하였으며, 다음과 같이 요약할 수 있다....
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10529310
   - 국문: 플렌옵틱은 물리공간 내 여러 방향의 빛 정보를 한꺼번에 센서에서 획득하고, 이를 그대로 재현함으로써 사용자에...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10559798
   - 국문: 네덜란드 스타트업 큐테크(QuTech)는 2018년 ‘Quantum internet: A vision for...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10536827
   - 국문: 본고에서는 양자암호 기술의 상용화를 위해 필요한 양자암호 네트워크 기술에 대해 알아본다. 최근 양자암호 기술...
   - 영문: 
----------------------------------------
🔹 NODE_ID: NODE10536828
   - 국문: 본고에서 우리는 고전

In [6]:
# SSU_Datathon2025_공학분야_62199_Increase_KYWD.json 에 추출된 키워드 추가

import json
import pandas as pd
import os

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
# (1) 원본 JSON 파일 (이전 단계에서 NaN 처리 및 URL 보정이 끝난 파일)
input_json_path = '../../SSU_Datathon2025_공학분야_62199.json'

# (2) 추출된 키워드 CSV 파일
input_csv_path = 'extract_keywords_from_abstract.csv'

# (3) 최종 결과 저장 파일명
output_json_path = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD.json'

def main():
    print("📂 데이터 병합 작업을 시작합니다...")

    # 1. CSV 파일 로드 (추출된 키워드)
    if not os.path.exists(input_csv_path):
        print(f"❌ 오류: CSV 파일({input_csv_path})이 없습니다.")
        return

    df = pd.read_csv(input_csv_path)
    
    # 매핑을 위해 딕셔너리로 변환 { 'NODE_ID': '키워드1, 키워드2' }
    # NaN 값을 빈 문자열로 변환 후 딕셔너리 생성
    df['EXTRACTED_KEYWORDS'] = df['EXTRACTED_KEYWORDS'].fillna("").astype(str)
    keyword_map = dict(zip(df['NODE_ID'], df['EXTRACTED_KEYWORDS']))
    
    print(f"   -> CSV 로드 완료: {len(keyword_map)}건의 매핑 정보")

    # 2. JSON 파일 로드
    if not os.path.exists(input_json_path):
        print(f"❌ 오류: JSON 파일({input_json_path})이 없습니다.")
        return

    with open(input_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 데이터 구조 확인 (NODE_LIST 키 존재 여부)
    if isinstance(data, dict) and "NODE_LIST" in data:
        node_list = data["NODE_LIST"]
    elif isinstance(data, list):
        node_list = data
    else:
        print("❌ JSON 데이터 구조를 인식할 수 없습니다.")
        return

    # 3. 키워드 병합 (기존 KYWD + 추출된 키워드)
    update_count = 0
    
    for node in node_list:
        node_id = node.get("NODE_ID")
        
        # 해당 논문의 추출된 키워드가 있는지 확인
        if node_id in keyword_map:
            new_keywords = keyword_map[node_id].strip()
            
            # 추가할 키워드가 빈 값이 아닌 경우에만 진행
            if new_keywords:
                original_keywords = str(node.get("KYWD", "")).strip()
                
                if original_keywords:
                    # 기존 키워드가 있으면 -> "기존, 신규" 형태로 연결
                    # (중복 방지를 위해 단순 연결보다는 set 등을 쓸 수도 있으나, 
                    #  여기서는 원본 보존을 위해 단순 연결 후 나중에 정제하는 방식을 추천합니다.
                    #  일단은 단순 연결로 처리합니다.)
                    node["KYWD"] = f"{original_keywords}, {new_keywords}"
                else:
                    # 기존 키워드가 없으면 -> "신규" 그대로 입력
                    node["KYWD"] = new_keywords
                
                update_count += 1

    # 4. 최종 저장
    final_output = {"NODE_LIST": node_list}
    
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_output, f, ensure_ascii=False, indent=4)

    print("\n" + "="*50)
    print(f"🎉 병합 완료!")
    print(f"- 총 처리된 논문: {len(node_list)}건")
    print(f"- 키워드가 추가/업데이트된 논문: {update_count}건")
    print(f"- 저장된 파일: {output_json_path}")
    print("="*50)

if __name__ == "__main__":
    main()

📂 데이터 병합 작업을 시작합니다...
   -> CSV 로드 완료: 679건의 매핑 정보

🎉 병합 완료!
- 총 처리된 논문: 62199건
- 키워드가 추가/업데이트된 논문: 575건
- 저장된 파일: SSU_Datathon2025_공학분야_62199_Increase_KYWD.json


In [7]:
# 원본 데이터에서 키워드 필드가 없는 논문 제목을 이용해 키워드 추출하는 작업

import pandas as pd
import json
import os
import re

# =============================================================================
# 1. 파일 경로 설정
# =============================================================================
input_json_path = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD.json'   # 원본 데이터 경로
rule_map_path = 'keyword_rule_map.json'                         # 룰맵 경로
output_json_path = 'SSU_Datathon2025_공학분야_62199_Increase_KYWD.json'              # [결과] 최종 저장될 JSON 파일명

# =============================================================================
# 2. 룰맵 로딩 및 정규화
# =============================================================================
print("1. 키워드 룰맵 로드 중...")
normalized_rule_map = {} 

try:
    with open(rule_map_path, 'r', encoding='utf-8') as f:
        rule_data = json.load(f)

    items = []
    if isinstance(rule_data, dict):
        if "exact_map" in rule_data:
            for v, c in rule_data["exact_map"].items():
                normalized_rule_map[v.replace(" ", "").lower()] = c
        if "variants_list" in rule_data:
            items = rule_data["variants_list"]
    elif isinstance(rule_data, list):
        items = rule_data

    for item in items:
        if "variant_norm" in item and "chosen" in item:
            normalized_rule_map[str(item["variant_norm"]).replace(" ", "").lower()] = item["chosen"]
    
    print(f"   -> 룰맵 준비 완료: {len(normalized_rule_map)}개 항목")

except FileNotFoundError:
    print(f"❌ 오류: 룰맵 파일({rule_map_path})을 찾을 수 없습니다.")
    exit()

# =============================================================================
# 3. 추출 함수 정의
# =============================================================================
COMMON_JOSA = ['은', '는', '이', '가', '을', '를', '의', '에', '와', '과', '로', '으로', '도', '만', '서']

def extract_keywords_from_text(text, rule_map, max_ngram=5):
    if not isinstance(text, str) or not text.strip():
        return ""

    clean_text = re.sub(r'[^\w\s가-힣]', ' ', text)
    words = clean_text.split()
    found_keywords = set()
    n = len(words)
    i = 0
    
    while i < n:
        match_found = False
        for length in range(min(max_ngram, n - i), 0, -1):
            chunk_words = words[i : i+length]
            base_chunk_str = "".join(chunk_words).lower()
            
            def check_and_add(search_key):
                if search_key in rule_map:
                    mapped_word = rule_map[search_key]
                    # 영어면 소문자 변환, 한글이면 유지
                    found_keywords.add(mapped_word.lower()) 
                    return True
                return False

            if check_and_add(base_chunk_str):
                i += length; match_found = True; break
            
            if len(base_chunk_str) > 1:
                last_char = base_chunk_str[-1]
                if last_char in COMMON_JOSA and check_and_add(base_chunk_str[:-1]):
                    i += length; match_found = True; break
                if len(base_chunk_str) > 2 and base_chunk_str.endswith('으로') and check_and_add(base_chunk_str[:-2]):
                    i += length; match_found = True; break

        if not match_found: i += 1
            
    return ",".join(list(found_keywords))

# =============================================================================
# 4. JSON 로드, 업데이트 및 저장
# =============================================================================
print("2. JSON 데이터 로드 및 처리 시작...")

if os.path.exists(input_json_path):
    # 1) 원본 JSON 읽기
    with open(input_json_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
    
    if "NODE_LIST" not in json_data:
        print("❌ JSON 구조 오류: 'NODE_LIST' 키를 찾을 수 없습니다.")
        exit()

    # 2) DataFrame 변환
    df = pd.DataFrame(json_data["NODE_LIST"])

    col_kywd = 'KYWD'
    col_title_kr = 'NODE_TTLE'
    col_title_en = 'NODE_TTLE_EN'

    # 3) 컬럼 전처리 (없으면 생성, NaN은 빈 문자열로)
    cols_to_check = [col_kywd, col_title_kr, col_title_en]
    for col in cols_to_check:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("").astype(str).str.strip()

    # 4) 대상 필터링: 키워드가 없는 경우
    target_indices = df[df[col_kywd] == ""].index
    print(f"   -> 총 {len(df)}건 중 키워드 없는 논문: {len(target_indices)}건")

    print("3. 키워드 추출 및 데이터 업데이트 중...")
    count = 0
    
    # 5) 키워드 추출 및 DataFrame 업데이트
    for idx in target_indices:
        title_text = df.at[idx, col_title_kr] + " " + df.at[idx, col_title_en]
        extracted_kwd = extract_keywords_from_text(title_text, normalized_rule_map)
        
        if extracted_kwd:
            df.at[idx, col_kywd] = extracted_kwd
            count += 1

    # =========================================================================
    # 5. 최종 JSON 저장 (핵심 추가 부분)
    # =========================================================================
    print("4. JSON 파일로 저장 중...")
    
    # 1) DataFrame을 다시 딕셔너리 리스트로 변환 (Records 형태)
    updated_node_list = df.to_dict(orient='records')
    
    # 2) 원본 JSON 객체의 NODE_LIST를 업데이트된 리스트로 교체
    json_data["NODE_LIST"] = updated_node_list
    
    # 3) 파일 쓰기
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, ensure_ascii=False, indent=4)

    print("-" * 50)
    print(f"✅ 모든 작업 완료!")
    print(f"   - 키워드 채움 성공: {count}건")
    print(f"   - 최종 결과 저장: {output_json_path}")
    print("-" * 50)
    
    # 결과 미리보기
    if count > 0:
        print("[업데이트 예시]")
        # 변경된 행 중 상위 3개만 찾아서 출력
        changed_df = df.loc[target_indices]
        changed_df = changed_df[changed_df[col_kywd] != ""]
        print(changed_df[[col_title_kr, col_kywd]].head(3))

else:
    print(f"❌ 오류: 입력 파일({input_json_path})이 없습니다.")

1. 키워드 룰맵 로드 중...
   -> 룰맵 준비 완료: 1931개 항목
2. JSON 데이터 로드 및 처리 시작...
   -> 총 62199건 중 키워드 없는 논문: 15836건
3. 키워드 추출 및 데이터 업데이트 중...
4. JSON 파일로 저장 중...
--------------------------------------------------
✅ 모든 작업 완료!
   - 키워드 채움 성공: 4800건
   - 최종 결과 저장: SSU_Datathon2025_공학분야_62199_Increase_KYWD.json
--------------------------------------------------
[업데이트 예시]
                     NODE_TTLE                                        KYWD
7   AI 및 IoT 기반 스마트 건물 자동제어시스템  internet of things,artificial intelligence
22   Submission Checklist etc.                                  submission
38    5G New Radio 기술 및 주파수 정책                                          5g
